In [1]:
import pandas as pd

# Load the nodes CSV file
nodes_df = pd.read_csv('/content/orfe_nodes_final_gravity.csv')

# Display the first 5 rows of the nodes DataFrame
print('First 5 rows of the nodes DataFrame:')
display(nodes_df.head())

First 5 rows of the nodes DataFrame:


,Id,Label,Layer,Affiliation,Subjects,Applications
0,1414099319,&#x00C2;ndrei Camponogara,2.0,"The Chinese University of Hong Kong, Shenzhen ...","['Optimization', 'Probability']","['Pure Theory', 'Networks/Telecommunications']"
1,1422426196,'. Gonz'alez,2.0,Princeton University (Inferred),"['Probability', 'Game Theory']","['Pure Theory', 'Public Policy']"
2,2226784417,'Alex Hern'andez-Garc'ia,2.0,"Mila Quebec AI Institute, Fraunhofer ITWM (Inf...",[],[]
3,2130393980,A. A.,2.0,Google (Inferred),[],[]
4,40804344,A. A. Abello,2.0,University of Texas at Austin (Inferred),[],[]


In [2]:
# Load the edges CSV file
edges_df = pd.read_csv('/content/orfe_edges_fuzzy_cleaned.csv')

# Display the first 5 rows of the edges DataFrame
print('\nFirst 5 rows of the edges DataFrame:')
display(edges_df.head())


First 5 rows of the edges DataFrame:


,Source,Target,Type,Weight
0,1678418,145967056,Undirected,3
1,1678595,2969311,Undirected,1
2,1678622,1696085,Undirected,1
3,1678622,1745169,Undirected,1
4,1678633,145944286,Undirected,1


In [ ]:
# Install networkx if not already installed
!pip install networkx

In [3]:
import networkx as nx
import pandas as pd
import html

# Ensure 'Id' and 'Source', 'Target' columns are of integer type
nodes_df['Id'] = nodes_df['Id'].astype(int)
edges_df['Source'] = edges_df['Source'].astype(int)
edges_df['Target'] = edges_df['Target'].astype(int)

# Clean up 'Layer' column: fill NaNs and convert to integer
nodes_df['Layer'] = nodes_df['Layer'].fillna(-1).astype(int) # Using -1 for unknown layer

# Decode HTML entities in 'Label' column for better readability
nodes_df['Label'] = nodes_df['Label'].apply(html.unescape)

# Create a mapping from node ID to its layer
node_to_layer = nodes_df.set_index('Id')['Layer'].to_dict()

In [4]:
# Create a graph
G = nx.Graph()

# Add nodes with their attributes from nodes_df
for index, row in nodes_df.iterrows():
    G.add_node(row['Id'], label=row['Label'], layer=row['Layer'], affiliation=row['Affiliation'], subjects=row['Subjects'], applications=row['Applications'])

# Add edges with their attributes from edges_df
for index, row in edges_df.iterrows():
    G.add_edge(row['Source'], row['Target'], type=row['Type'], weight=row['Weight'])

In [5]:
# Calculate degrees for all nodes
degrees = dict(G.degree())

# Initialize dictionaries to store degrees for each layer
degrees_by_layer = {
    0: [],
    1: [],
    2: []
}

# Populate degrees for each layer based on the node_to_layer mapping
for node_id, degree in degrees.items():
    layer = node_to_layer.get(node_id) # Get layer of the node
    if layer in degrees_by_layer:
        degrees_by_layer[layer].append(degree)

# Calculate and print the average degree for each specified layer
for layer, deg_list in degrees_by_layer.items():
    if deg_list:
        average_degree = sum(deg_list) / len(deg_list)
        print(f"Average degree for nodes in Layer {layer}: {average_degree:.2f}")
    else:
        print(f"No nodes found in Layer {layer} to calculate average degree.")

Average degree for nodes in Layer 0: 40.47
Average degree for nodes in Layer 1: 58.61
Average degree for nodes in Layer 2: 1.28


In [6]:
# Count the number of nodes in each layer
nodes_per_layer = nodes_df['Layer'].value_counts().sort_index()

print("Number of nodes in each layer:")
for layer in [0, 1, 2]:
    count = nodes_per_layer.get(layer, 0)
    print(f"Layer {layer}: {count} nodes")

Number of nodes in each layer:
Layer 0: 17 nodes
Layer 1: 550 nodes
Layer 2: 22843 nodes


In [7]:
# Calculate the average degree of the entire graph
overall_average_degree = sum(degrees.values()) / len(degrees)
print(f"Average degree of the entire graph: {overall_average_degree:.2f}")

Average degree of the entire graph: 2.65


In [8]:
import re

def clean_affiliation(affiliation_str):
    if pd.isna(affiliation_str):
        return None
    # Remove anything in parentheses at the end (e.g., '(Inferred)')
    cleaned = re.sub(r'\s*\(.*\)\s*$', '', affiliation_str)
    # Take the part before the first comma, if any
    cleaned = cleaned.split(',')[0]
    return cleaned.strip()

In [9]:
# Apply the cleaning function to create a new 'Cleaned_Affiliation' column
nodes_df['Cleaned_Affiliation'] = nodes_df['Affiliation'].apply(clean_affiliation)

# Get the top 5 most frequent affiliations
top_affiliations = nodes_df['Cleaned_Affiliation'].value_counts().head(5).index.tolist()

# Add specific universities requested by the user if they are not already in the top 5
# We'll use the cleaned names for better matching
user_specific_universities = ['Princeton University', 'Stanford University', 'MIT']
for uni in user_specific_universities:
    if uni not in top_affiliations:
        # Check if the cleaned affiliation contains the university name (case-insensitive and partial match)
        if nodes_df['Cleaned_Affiliation'].str.contains(uni, case=False, na=False).any():
             # Find the exact cleaned name if it exists, otherwise use the provided name
            exact_match = nodes_df['Cleaned_Affiliation'][nodes_df['Cleaned_Affiliation'].str.contains(uni, case=False, na=False)].iloc[0]
            if exact_match not in top_affiliations:
                top_affiliations.append(exact_match)
        else:
            # If no node exists with a cleaned affiliation containing the user-specified university, just add it for now
            # This might result in 0 nodes/edges if the name isn't present in the data
            top_affiliations.append(uni)

print("Analyzing the following top affiliations:", top_affiliations)

print("\n--- Analysis per Affiliation ---")
# Dictionary to store results
affiliation_stats = {}

for affiliation_name in top_affiliations:
    # Get nodes belonging to the current affiliation
    current_affiliation_nodes_df = nodes_df[nodes_df['Cleaned_Affiliation'] == affiliation_name]
    num_nodes = len(current_affiliation_nodes_df)

    # Get the IDs of these nodes
    current_affiliation_node_ids = current_affiliation_nodes_df['Id'].tolist()

    # Count edges where both source and target are within this affiliation
    num_edges = edges_df[
        edges_df['Source'].isin(current_affiliation_node_ids) &
        edges_df['Target'].isin(current_affiliation_node_ids)
    ].shape[0]

    affiliation_stats[affiliation_name] = {'nodes': num_nodes, 'edges': num_edges}
    print(f"{affiliation_name}: {num_nodes} nodes, {num_edges} edges")

Analyzing the following top affiliations: ['Princeton University', 'Stanford University', 'Hong Kong University of Science and Technology', 'Google', 'Shanghai University of Finance and Economics', 'MIT']

--- Analysis per Affiliation ---
Princeton University: 6175 nodes, 7514 edges
Stanford University: 3837 nodes, 3821 edges
Hong Kong University of Science and Technology: 1473 nodes, 1471 edges
Google: 1324 nodes, 1344 edges
Shanghai University of Finance and Economics: 1303 nodes, 1311 edges
MIT: 65 nodes, 59 edges


In [10]:
print('\n--- Graph Connectivity Analysis ---')

# 1. Check if the graph is connected
is_connected = nx.is_connected(G)
print(f"Is the graph fully connected? {is_connected}")


--- Graph Connectivity Analysis ---
Is the graph fully connected? False


In [11]:
# 2. Find all connected components
components = list(nx.connected_components(G))
num_components = len(components)
print(f"Number of connected components: {num_components}")

# 3. Find the giant component (the largest one)
if components:
    giant_component = max(components, key=len)
    num_nodes_in_giant_component = len(giant_component)

    # 4. Count nodes not in the giant component
    num_nodes_not_in_giant_component = G.number_of_nodes() - num_nodes_in_giant_component

    print(f"Number of nodes in the giant component: {num_nodes_in_giant_component}")
    print(f"Number of nodes not in the giant component: {num_nodes_not_in_giant_component}")
else:
    print("The graph has no components (it might be empty).")

Number of connected components: 2
Number of nodes in the giant component: 23409
Number of nodes not in the giant component: 1


In [12]:
# The unconnected node ID was identified from the components analysis
unconnected_node_id = 102254552

# Retrieve the label of this node from the nodes_df
unconnected_node_label = nodes_df[nodes_df['Id'] == unconnected_node_id]['Label'].iloc[0]

print(f"The unconnected node is: ID {unconnected_node_id}, Label: '{unconnected_node_label}'")

The unconnected node is: ID 102254552, Label: 'Ludovic Tangpi Ndounkeu'


In [13]:
# Initialize a counter for stars and a list to store their hub IDs
num_stars = 0
star_hubs = []

# Filter nodes that are in Layer 1, as these are potential 'hub' nodes for a star
layer1_nodes_df = nodes_df[nodes_df['Layer'] == 1]

# Iterate through each potential hub node in Layer 1
for hub_id in layer1_nodes_df['Id']:
    # Get all neighbors of the current potential hub node
    neighbors = list(G.neighbors(hub_id))

    # Condition 1: A hub node must have connections (spokes)
    if not neighbors:
        continue

    # Condition 2: All neighbors of the Layer 1 hub node must be in Layer 2
    # And the hub should not be connected to any other Layer 1 nodes or Layer 0 nodes
    is_potential_hub = True
    layer2_spokes = [] # To store the Layer 2 neighbors of this hub
    for neighbor_id in neighbors:
        neighbor_layer = node_to_layer.get(neighbor_id)
        if neighbor_layer == 2: # Neighbor is in Layer 2, so it's a valid spoke candidate
            layer2_spokes.append(neighbor_id)
        else: # Neighbor is in Layer 0, Layer 1, or an unexpected layer, so it's not a valid hub
            is_potential_hub = False
            break

    # If the hub isn't valid, or it has no Layer 2 spokes, move to the next potential hub
    if not is_potential_hub or not layer2_spokes:
        continue

    # Condition 3: Now, check if each Layer 2 spoke node is ONLY connected back to this hub
    is_valid_star = True
    for spoke_id in layer2_spokes:
        spoke_neighbors = list(G.neighbors(spoke_id))
        # The spoke node must have exactly one connection, and that connection must be to the current hub
        if not (len(spoke_neighbors) == 1 and spoke_neighbors[0] == hub_id):
            is_valid_star = False
            break

    # If all conditions are met, increment the star count and record the hub ID
    if is_valid_star:
        num_stars += 1
        star_hubs.append(hub_id)

print(f"Number of stars found in the graph: {num_stars}")
print(f"IDs of the Layer 1 hub nodes that form stars: {star_hubs}")

Number of stars found in the graph: 0
IDs of the Layer 1 hub nodes that form stars: []


In [14]:
# Initialize a counter for stars and a list to store their hub IDs for the new layer mapping
num_stars_0_1 = 0
star_hubs_0_1 = []

# Filter nodes that are in Layer 0, as these are potential 'hub' nodes for this star definition
layer0_nodes_df = nodes_df[nodes_df['Layer'] == 0]

print("\n--- Star Detection (Layer 0 as Hubs, Layer 1 as Spokes) ---")

# Iterate through each potential hub node in Layer 0
for hub_id in layer0_nodes_df['Id']:
    # Get all neighbors of the current potential hub node
    neighbors = list(G.neighbors(hub_id))

    # Condition 1: A hub node must have connections (spokes)
    if not neighbors:
        continue

    # Condition 2: All neighbors of the Layer 0 hub node must be in Layer 1
    # And the hub should not be connected to any other Layer 0 nodes or Layer 2 nodes
    is_potential_hub = True
    layer1_spokes = [] # To store the Layer 1 neighbors of this hub
    for neighbor_id in neighbors:
        neighbor_layer = node_to_layer.get(neighbor_id)
        if neighbor_layer == 1: # Neighbor is in Layer 1, so it's a valid spoke candidate
            layer1_spokes.append(neighbor_id)
        else: # Neighbor is in Layer 0, Layer 2, or an unexpected layer, so it's not a valid hub
            is_potential_hub = False
            break

    # If the hub isn't valid, or it has no Layer 1 spokes, move to the next potential hub
    if not is_potential_hub or not layer1_spokes:
        continue

    # Condition 3: Now, check if each Layer 1 spoke node is ONLY connected back to this hub
    is_valid_star = True
    for spoke_id in layer1_spokes:
        spoke_neighbors = list(G.neighbors(spoke_id))
        # The spoke node must have exactly one connection, and that connection must be to the current hub
        if not (len(spoke_neighbors) == 1 and spoke_neighbors[0] == hub_id):
            is_valid_star = False
            break

    # If all conditions are met, increment the star count and record the hub ID
    if is_valid_star:
        num_stars_0_1 += 1
        star_hubs_0_1.append(hub_id)

print(f"Number of stars found (Layer 0 hubs, Layer 1 spokes): {num_stars_0_1}")
print(f"IDs of the Layer 0 hub nodes that form these stars: {star_hubs_0_1}")


--- Star Detection (Layer 0 as Hubs, Layer 1 as Spokes) ---
Number of stars found (Layer 0 hubs, Layer 1 spokes): 0
IDs of the Layer 0 hub nodes that form these stars: []


In [15]:
# Initialize a counter for stars and a list to store their hub IDs for this new definition
num_stars_modified_def = 0
star_hubs_modified_def = []

# Filter nodes that are in Layer 1, as these are potential 'hub' nodes
layer1_nodes_df = nodes_df[nodes_df['Layer'] == 1]

print("\n--- Star Detection (Modified Definition: Layer 1 hubs, Layer 2 spokes, L1-L1 connections allowed) ---")

# Iterate through each potential hub node in Layer 1
for hub_id in layer1_nodes_df['Id']:
    # Get all neighbors of the current potential hub node
    all_neighbors = list(G.neighbors(hub_id))

    # Identify neighbors that are in Layer 2 (these are the potential spokes)
    layer2_spokes_candidates = [
        neighbor_id for neighbor_id in all_neighbors
        if node_to_layer.get(neighbor_id) == 2
    ]

    # Condition 1: A Layer 1 hub must have at least one Layer 2 spoke to form a star
    if not layer2_spokes_candidates:
        continue # This Layer 1 node has no Layer 2 neighbors, so it can't be a hub in this definition

    # Condition 2: Now, check if each identified Layer 2 spoke candidate is ONLY connected back to this hub
    is_valid_star = True
    for spoke_id in layer2_spokes_candidates:
        spoke_neighbors = list(G.neighbors(spoke_id))
        # The spoke node must have exactly one connection, and that connection must be to the current hub
        if not (len(spoke_neighbors) == 1 and spoke_neighbors[0] == hub_id):
            is_valid_star = False
            break

    # If all conditions are met for all its Layer 2 neighbors, increment the star count and record the hub ID
    if is_valid_star:
        num_stars_modified_def += 1
        star_hubs_modified_def.append(hub_id)

print(f"Number of stars found (Modified Definition): {num_stars_modified_def}")
print(f"IDs of the Layer 1 hub nodes that form these stars: {star_hubs_modified_def}")


--- Star Detection (Modified Definition: Layer 1 hubs, Layer 2 spokes, L1-L1 connections allowed) ---
Number of stars found (Modified Definition): 44
IDs of the Layer 1 hub nodes that form these stars: [46817751, 2335761675, 32312258, 2028197361, 143902023, 47199866, 144720027, 1803012, 2064165512, 12611133, 2178999563, 2273375130, 11308925, 119066377, 2387972597, 2284064769, 66119414, 2154235062, 113747981, 2310030338, 2238954007, 2258520575, 121494824, 52044140, 2179477, 2052717147, 27248138, 16120045, 46866436, 2367856237, 1392620177, 2158592259, 88740336, 2312767231, 2163392214, 9823600, 3046493, 77579521, 8904571, 2335557622, 2394361569, 2329512190, 2243153615, 47196706]


In [16]:
import pandas as pd

# Create a list to store star details
star_details = []

for hub_id in star_hubs_modified_def:
    # Get all neighbors of the current hub node
    all_neighbors = list(G.neighbors(hub_id))

    # Identify neighbors that are in Layer 2 (these are the spokes)
    layer2_spokes_for_hub = [
        neighbor_id for neighbor_id in all_neighbors
        if node_to_layer.get(neighbor_id) == 2
    ]

    # The size of the star is the number of Layer 2 spokes
    star_size = len(layer2_spokes_for_hub)

    # Get the professor's name (label) for the hub ID
    professor_name = nodes_df[nodes_df['Id'] == hub_id]['Label'].iloc[0]

    star_details.append({
        'Hub ID': hub_id,
        'Professor Name': professor_name,
        'Star Size (L2 Spokes)': star_size
    })

# Convert to DataFrame for easier sorting and display
star_details_df = pd.DataFrame(star_details)

print("\n--- Stars Analysis ---")

# Sort by star size to find the largest and smallest
sorted_stars = star_details_df.sort_values(by='Star Size (L2 Spokes)', ascending=False)

print("\nTop 3 Largest Stars:")
display(sorted_stars.head(3))

print("\nBottom 3 Smallest Stars:")
display(sorted_stars.tail(3))


--- Stars Analysis ---

Top 3 Largest Stars:


,Hub ID,Professor Name,Star Size (L2 Spokes)
9,12611133,C. Maes,120
8,2064165512,Burak Aydın,26
32,88740336,R. Titiunik,25



Bottom 3 Smallest Stars:


,Hub ID,Professor Name,Star Size (L2 Spokes)
20,2238954007,Jackie Lok,1
34,2163392214,Rajita Chandak,1
40,2394361569,Toby Anderson,1


In [ ]:
total_l2_spokes = star_details_df['Star Size (L2 Spokes)'].sum()
print(f"Total sum of L2 spokes across all identified stars: {total_l2_spokes}")

Total sum of L2 spokes across all identified stars: 377


In [17]:
# Initialize a counter for Layer 2 nodes with degree 1
layer2_degree_1_nodes_count = 0

# Iterate through all nodes and their degrees
for node_id, degree_val in degrees.items():
    # Check if the node is in Layer 2 and its degree is 1
    if node_to_layer.get(node_id) == 2 and degree_val == 1:
        layer2_degree_1_nodes_count += 1

print(f"Total number of Layer 2 nodes with an outdegree of 1: {layer2_degree_1_nodes_count}")

Total number of Layer 2 nodes with an outdegree of 1: 18804


In [18]:
import networkx as nx
from collections import Counter

print("\n--- Cycle Analysis ---")

# Find a cycle basis for the graph
# Note: For large graphs, finding all simple cycles (nx.simple_cycles) can be extremely slow.
# nx.cycle_basis finds a fundamental cycle basis, which is a more manageable set of cycles.
critical_nodes = [node for node, degree in G.degree() if degree > 0]
G_sub = G.subgraph(critical_nodes)



cycles = nx.cycle_basis(G_sub)

# Total number of cycles found
total_cycles = len(cycles)
print(f"Total number of cycles (from cycle basis): {total_cycles}")

# Count cycles of each length
cycle_lengths = [len(cycle) for cycle in cycles]
cycle_length_counts = Counter(cycle_lengths)

print("Number of cycles of each length:")
for length, count in sorted(cycle_length_counts.items()):
    print(f"  Length {length}: {count} cycles")


--- Cycle Analysis ---
Total number of cycles (from cycle basis): 7621
Number of cycles of each length:
  Length 3: 4263 cycles
  Length 4: 315 cycles
  Length 5: 183 cycles
  Length 6: 85 cycles
  Length 7: 91 cycles
  Length 8: 57 cycles
  Length 9: 68 cycles
  Length 10: 53 cycles
  Length 11: 30 cycles
  Length 12: 51 cycles
  Length 13: 49 cycles
  Length 14: 68 cycles
  Length 15: 57 cycles
  Length 16: 63 cycles
  Length 17: 39 cycles
  Length 18: 36 cycles
  Length 19: 51 cycles
  Length 20: 49 cycles
  Length 21: 51 cycles
  Length 22: 22 cycles
  Length 23: 32 cycles
  Length 24: 38 cycles
  Length 25: 35 cycles
  Length 26: 38 cycles
  Length 27: 20 cycles
  Length 28: 39 cycles
  Length 29: 72 cycles
  Length 30: 60 cycles
  Length 31: 53 cycles
  Length 32: 49 cycles
  Length 33: 42 cycles
  Length 34: 22 cycles
  Length 35: 58 cycles
  Length 36: 47 cycles
  Length 37: 53 cycles
  Length 38: 49 cycles
  Length 39: 46 cycles
  Length 40: 41 cycles
  Length 41: 38 cycles
 

### Analyzing Applications and Subjects

First, we need to process the 'Subjects' and 'Applications' columns, as they contain string representations of lists. We'll extract individual subjects and applications to count their occurrences.

In [20]:
import ast # Used for safely evaluating string representations of lists

# Function to safely parse string representations of lists or single strings
def parse_list_string(s):
    if pd.isna(s):
        return []
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return [s]

# Apply the parsing function to 'Subjects' and 'Applications'
nodes_df['Parsed_Subjects'] = nodes_df['Subjects'].apply(parse_list_string)
nodes_df['Parsed_Applications'] = nodes_df['Applications'].apply(parse_list_string)

# Flatten the lists and count occurrences for Subjects
all_subjects = [item for sublist in nodes_df['Parsed_Subjects'] for item in sublist]
subject_counts = pd.Series(all_subjects).value_counts()

# Flatten the lists and count occurrences for Applications
all_applications = [item for sublist in nodes_df['Parsed_Applications'] for item in sublist]
application_counts = pd.Series(all_applications).value_counts()

print("\n--- All Subject Counts ---")
display(subject_counts)

print("\n--- All Application Counts ---")
display(application_counts)


--- All Subject Counts ---


,count
Probability,21753
Statistics and Machine Learning (ML),10068
Optimization,6503
Game Theory,5307
Financial Engineering,329



--- All Application Counts ---


,count
Pure Theory,12538
Public Policy,10535
Natural Language Processing,5043
Healthcare,4400
Networks/Telecommunications,4160
Robotics,3139
Energy,2991
Transportation/Logistics,560
Finance,504
Supply Chain/Revenue Management,90


In [21]:
total_subject_mentions = subject_counts.sum()
total_application_mentions = application_counts.sum()

print(f"Total mentions across all subjects: {total_subject_mentions}")
print(f"Total mentions across all applications: {total_application_mentions}")

Total mentions across all subjects: 43960
Total mentions across all applications: 43960


### Statistics for Top and Bottom Applications/Subjects

Now, let's look at more statistics for these common and uncommon categories, including the number of nodes, average degree, and internal edges within each group.

In [ ]:
# Helper function to get stats for a given attribute type and value
def get_stats_for_attribute(attribute_type, attribute_value):
    # Filter nodes_df based on the attribute value
    if attribute_type == 'Subjects':
        filtered_nodes_df = nodes_df[nodes_df['Parsed_Subjects'].apply(lambda x: attribute_value in x)]
    elif attribute_type == 'Applications':
        filtered_nodes_df = nodes_df[nodes_df['Parsed_Applications'].apply(lambda x: attribute_value in x)]
    else:
        return None

    node_ids = filtered_nodes_df['Id'].tolist()
    num_nodes = len(node_ids)

    # Calculate average degree for nodes in this group
    if node_ids:
        avg_degree = sum(G.degree(n) for n in node_ids) / num_nodes
    else:
        avg_degree = 0

    # Count internal edges (edges where both endpoints are in this group)
    internal_edges = 0
    for u, v in G.edges():
        if u in node_ids and v in node_ids:
            internal_edges += 1

    return {'nodes': num_nodes, 'avg_degree': avg_degree, 'internal_edges': internal_edges}

# Get top 3 subjects and their statistics
top_3_subjects = subject_counts.head(3).index.tolist()
print("\n--- Top 3 Subject Statistics ---")
for subject in top_3_subjects:
    stats = get_stats_for_attribute('Subjects', subject)
    print(f"Subject: '{subject}' - Nodes: {stats['nodes']}, Avg Degree: {stats['avg_degree']:.2f}, Internal Edges: {stats['internal_edges']}")

# Get bottom 3 subjects and their statistics
bottom_3_subjects = subject_counts.tail(3).index.tolist()
print("\n--- Bottom 3 Subject Statistics ---")
for subject in bottom_3_subjects:
    stats = get_stats_for_attribute('Subjects', subject)
    print(f"Subject: '{subject}' - Nodes: {stats['nodes']}, Avg Degree: {stats['avg_degree']:.2f}, Internal Edges: {stats['internal_edges']}")

# Get top 3 applications and their statistics
top_3_applications = application_counts.head(3).index.tolist()
print("\n--- Top 3 Application Statistics ---")
for app in top_3_applications:
    stats = get_stats_for_attribute('Applications', app)
    print(f"Application: '{app}' - Nodes: {stats['nodes']}, Avg Degree: {stats['avg_degree']:.2f}, Internal Edges: {stats['internal_edges']}")

# Get bottom 3 applications and their statistics
bottom_3_applications = application_counts.tail(3).index.tolist()
print("\n--- Bottom 3 Application Statistics ---")
for app in bottom_3_applications:
    stats = get_stats_for_attribute('Applications', app)
    print(f"Application: '{app}' - Nodes: {stats['nodes']}, Avg Degree: {stats['avg_degree']:.2f}, Internal Edges: {stats['internal_edges']}")


--- Top 3 Subject Statistics ---
Subject: 'Probability' - Nodes: 21753, Avg Degree: 2.53, Internal Edges: 24364
Subject: 'Statistics and Machine Learning (ML)' - Nodes: 10068, Avg Degree: 3.59, Internal Edges: 10853
Subject: 'Optimization' - Nodes: 6503, Avg Degree: 2.36, Internal Edges: 2400

--- Bottom 3 Subject Statistics ---
Subject: 'Optimization' - Nodes: 6503, Avg Degree: 2.36, Internal Edges: 2400
Subject: 'Game Theory' - Nodes: 5307, Avg Degree: 2.35, Internal Edges: 1799
Subject: 'Financial Engineering' - Nodes: 329, Avg Degree: 2.81, Internal Edges: 74

--- Top 3 Application Statistics ---
Application: 'Pure Theory' - Nodes: 12538, Avg Degree: 3.36, Internal Edges: 14610
Application: 'Public Policy' - Nodes: 10535, Avg Degree: 2.41, Internal Edges: 6228
Application: 'Natural Language Processing' - Nodes: 5043, Avg Degree: 3.03, Internal Edges: 2441

--- Bottom 3 Application Statistics ---
Application: 'Transportation/Logistics' - Nodes: 560, Avg Degree: 2.37, Internal Edges

### Applications and Subjects Most Involved in 3-Cycles

Finally, let's identify which applications and subjects are most frequently part of 3-cycles, which can indicate tightly knit research areas or collaborations.

In [ ]:
import collections

# Create mappings from node ID to its parsed subjects and applications for efficient lookup
node_id_to_subjects = nodes_df.set_index('Id')['Parsed_Subjects'].to_dict()
node_id_to_applications = nodes_df.set_index('Id')['Parsed_Applications'].to_dict()

# Counters for subjects and applications involved in 3-cycles
subject_cycle_counts = collections.Counter()
application_cycle_counts = collections.Counter()

# Iterate through each 3-cycle found previously
for cycle in cycles_len_3:
    involved_subjects_in_cycle = set() # Use a set to count each subject/app only once per cycle
    involved_applications_in_cycle = set()
    for node_id in cycle:
        # Get subjects/applications for this node, default to empty list if not found
        subjects = node_id_to_subjects.get(node_id, [])
        applications = node_id_to_applications.get(node_id, [])

        # Add them to the sets to avoid double counting within the same cycle
        involved_subjects_in_cycle.update(subjects)
        involved_applications_in_cycle.update(applications)

    # Increment overall counts for each unique subject/application found in this cycle
    for subject in involved_subjects_in_cycle:
        subject_cycle_counts[subject] += 1
    for app in involved_applications_in_cycle:
        application_cycle_counts[app] += 1

print("\n--- Subjects Most Involved in 3-Cycles (Top 5) ---")
display(subject_cycle_counts.most_common(5))

print("\n--- Applications Most Involved in 3-Cycles (Top 5) ---")
display(application_cycle_counts.most_common(5))


--- Subjects Most Involved in 3-Cycles (Top 5) ---


[('Probability', 4263),
 ('Statistics and Machine Learning (ML)', 3888),
 ('Optimization', 2245),
 ('Game Theory', 1524),
 ('Financial Engineering', 105)]


--- Applications Most Involved in 3-Cycles (Top 5) ---


[('Pure Theory', 3839),
 ('Public Policy', 3128),
 ('Natural Language Processing', 2325),
 ('Networks/Telecommunications', 1926),
 ('Robotics', 1412)]

In [ ]:
# Filter nodes where 'Optimization' and 'Game Theory' are both in 'Parsed_Subjects'
optimization_game_theory_specialists = nodes_df[
    nodes_df['Parsed_Subjects'].apply(lambda x: 'Optimization' in x and 'Game Theory' in x)
]

num_specialists = len(optimization_game_theory_specialists)

print(f"Number of people specializing in both Optimization and Game Theory: {num_specialists}")

# Optionally, display the first few rows of these specialists
# display(optimization_game_theory_specialists.head())

Number of people specializing in both Optimization and Game Theory: 30


### Graph Metrics Analysis: Centrality Scores

Now, let's load the new graph metrics dataset and analyze the top and bottom 5 nodes for each type of centrality.

In [ ]:
import pandas as pd

# Load the graph metrics CSV file
metrics_df = pd.read_csv('/content/orfe_graph_metrics.csv')

print('First 5 rows of the graph metrics DataFrame:')
display(metrics_df.head())

# Define the centrality columns to analyze based on user request and available data
requested_centralities = {
    'degree_x': 'degree_centrality_x',
    'degree_y': 'degree_centrality_y',
    'degree': 'degree_centrality',
    'betweenness': 'betweenness_centrality',
    'closeness': 'closeness_centrality' # This column might not exist
}

centrality_columns_to_process = []
missing_centralities = []

for name, col_name in requested_centralities.items():
    if col_name in metrics_df.columns:
        centrality_columns_to_process.append(col_name)
    else:
        missing_centralities.append(name)

if missing_centralities:
    print(f"\nWarning: The following centrality metrics were requested but not found in the DataFrame: {', '.join(missing_centralities)}.")
    print(f"Available columns in metrics_df: {metrics_df.columns.tolist()}")

if not centrality_columns_to_process:
    print("No valid centrality columns found in the DataFrame to analyze.")
elif 'Label' not in metrics_df.columns or 'vertex' not in metrics_df.columns:
    print("Error: 'Label' or 'vertex' column not found in metrics_df. These are required for display.")
    print(f"Available columns in metrics_df: {metrics_df.columns.tolist()}")
else:
    print(f"\nAnalyzing available centrality columns: {centrality_columns_to_process}")
    print('\n--- Top 10 and Bottom 10 Centrality Scores ---')

    for col in centrality_columns_to_process:
        print(f"\nAnalyzing: {col}")

        # Top 10 highest centrality scores
        top_10 = metrics_df.nlargest(10, col)
        print(f"Top 10 highest {col}:")
        display(top_10[['vertex', 'Label', col]])

        # Bottom 10 lowest centrality scores
        # Showing nodes with the absolute lowest scores, which could be 0
        bottom_10 = metrics_df.nsmallest(10, col)
        print(f"Bottom 10 lowest {col}:")
        display(bottom_10[['vertex', 'Label', col]])

First 5 rows of the graph metrics DataFrame:


,vertex,Label,degree_centrality_x,degree_centrality_y,degree_centrality
0,47829900,Fan Yang,0.181469,0.394815,0.382655
1,38144094,Tiangang Zhang,0.073903,0.353783,0.135495
2,145784853,Xiao Han,0.065573,0.348931,0.118172
3,49596260,Surinder Kumar,0.058012,0.304501,0.111418
4,145967056,H. Poor,0.031569,0.332992,0.069674



Available columns in metrics_df: ['vertex', 'Label', 'degree_centrality_x', 'degree_centrality_y', 'degree_centrality']

Analyzing available centrality columns: ['degree_centrality_x', 'degree_centrality_y', 'degree_centrality']

--- Top 10 and Bottom 10 Centrality Scores ---

Analyzing: degree_centrality_x
Top 10 highest degree_centrality_x:


,vertex,Label,degree_centrality_x
0,47829900,Fan Yang,0.181469
1,38144094,Tiangang Zhang,0.073903
2,145784853,Xiao Han,0.065573
3,49596260,Surinder Kumar,0.058012
4,145967056,H. Poor,0.031569
7,2969311,Zhangyang Wang,0.022427
9,49988044,Joseph E. Gonzalez,0.017985
12,144344283,Ken Goldberg,0.016788
11,145944286,Daniela Rus,0.016105
14,2258678973,Dawn Xiaodong Song,0.014866


Bottom 10 lowest degree_centrality_x:


...

,vertex,Label,degree_centrality_y
20050,102254552,Ludovic Tangpi Ndounkeu,0.000000
3203,145801247,S. Hajji,0.154990
5465,2072732873,P. Takam,0.155015
5823,2187428817,Rhoss Likibi Pellat,0.155015
6021,144391654,Ralf Wunderlich,0.155015
10177,1856101,V. K. Socgnia,0.155015
10229,51124586,V. Djeundje,0.155015
12337,2398778471,Wilfried Kuissi Kamdem,0.155015
13431,2295675605,Dai Taguchi,0.155015
14067,145881085,G. Reis,0.155015



Analyzing: degree_centrality
Top 10 highest degree_centrality:


,vertex,Label,degree_centrality
0,47829900,Fan Yang,0.382655
1,38144094,Tiangang Zhang,0.135495
2,145784853,Xiao Han,0.118172
3,49596260,Surinder Kumar,0.111418
4,145967056,H. Poor,0.069674
5,8386952,Bartolomeo Stellato,0.052972
6,2266124380,Jianqing Fan,0.051907
7,2969311,Zhangyang Wang,0.043119
8,2258225151,Boris Hanin,0.041618
9,49988044,Joseph E. Gonzalez,0.033971


Bottom 10 lowest degree_centrality:


,vertex,Label,degree_centrality
2596,48876151,Mingyu Ding,0.0
2597,1581341133,S. Cohan,0.0
2598,2005667112,S. Chabra,0.0
2599,1712880,S. Tomasin,0.0
2600,2404961226,Ruixuan Li,0.0
2601,14256521,S. Chun,0.0
2602,144071961,S. Tiwari,0.0
2603,2349956736,Ryan Zhao,0.0
2604,153671941,S. Chander,0.0
2605,108554478,S. Chua,0.0


### Degree Centrality for Subgraph (Layer 0 and Layer 1 nodes only)

In [ ]:
# Identify nodes belonging to Layer 0 or Layer 1
layer0_1_nodes = [node_id for node_id, layer in node_to_layer.items() if layer in [0, 1]]

# Create a subgraph with only these nodes
G_sub_l0_l1 = G.subgraph(layer0_1_nodes)

# Calculate degree centrality for the subgraph
degree_centrality_l0_l1 = nx.degree_centrality(G_sub_l0_l1)

# Convert to a DataFrame for easier sorting and display
degree_centrality_l0_l1_df = pd.DataFrame(
    list(degree_centrality_l0_l1.items()), columns=['vertex', 'degree_centrality_l0_l1']
)

# Merge with original nodes_df to get labels
degree_centrality_l0_l1_df = degree_centrality_l0_l1_df.merge(
    nodes_df[['Id', 'Label']], left_on='vertex', right_on='Id', how='left'
).drop(columns=['Id'])

print('\n--- Top 10 and Bottom 10 Degree Centrality for Layer 0 & 1 Subgraph ---')

# Top 10 highest degree centrality
top_10_l0_l1 = degree_centrality_l0_l1_df.nlargest(10, 'degree_centrality_l0_l1')
print("Top 10 highest degree centrality (Layer 0 & 1):")
display(top_10_l0_l1[['vertex', 'Label', 'degree_centrality_l0_l1']])

# Bottom 10 lowest degree centrality
bottom_10_l0_l1 = degree_centrality_l0_l1_df.nsmallest(10, 'degree_centrality_l0_l1')
print("Bottom 10 lowest degree centrality (Layer 0 & 1):")
display(bottom_10_l0_l1[['vertex', 'Label', 'degree_centrality_l0_l1']])


--- Top 10 and Bottom 10 Degree Centrality for Layer 0 & 1 Subgraph ---
Top 10 highest degree centrality (Layer 0 & 1):


,vertex,Label,degree_centrality_l0_l1
26,2266124380,Jianqing Fan,0.210247
117,8386952,Bartolomeo Stellato,0.132509
566,2258225151,Boris Hanin,0.116608
253,12890978,E. Rebrova,0.086572
168,7402018,R. Sircar,0.072438
459,1697413,Sanjeev R. Kulkarni,0.070671
339,9051324,Jason M. Klusowski,0.068905
266,47829900,Fan Yang,0.061837
327,25681050,Matias D. Cattaneo,0.061837
426,1825721854,H. Mete Soner,0.056537


Bottom 10 lowest degree centrality (Layer 0 & 1):


,vertex,Label,degree_centrality_l0_l1
556,102254552,Ludovic Tangpi Ndounkeu,0.000000
41,144316575,Kevin Webster,0.001767
64,2384763134,Alexander Zlokapa,0.001767
76,2001172,J. Escanciano,0.001767
79,51917083,P. J. Graber,0.001767
93,12147009,M. Sena,0.001767
113,30007671,Stephen Jiang,0.001767
119,153082259,Qin-Rong Yan,0.001767
159,2258520575,Joyce Luo,0.001767
174,152824371,J. Chu,0.001767


### Analysis of Normalized Degree Centrality Y

Now, let's calculate a new metric by dividing the `degree_centrality_y` by the raw number of edges connected to each node (its degree). This helps to understand centrality relative to a node's immediate connections. We'll then look at the top and bottom performers for this new metric, and analyze how rankings change compared to the original `degree_centrality_y`.

In [ ]:
# Create a temporary DataFrame for calculation, merging with degrees
temp_df = metrics_df[['vertex', 'Label', 'degree_centrality_y', 'degree_centrality']].copy()

# Add raw degree to this temporary DataFrame (already available if 'degrees' is in kernel state)
# temp_df['raw_degree'] = temp_df['vertex'].map(degrees) # This line is not needed for the new calculation

# Calculate the new metric: degree centrality y divided by overall degree centrality
# Handle cases where degree_centrality is 0 to avoid division by zero
temp_df['normalized_degree_centrality_y'] = temp_df.apply(
    lambda row: row['degree_centrality_y'] / row['degree_centrality'] if row['degree_centrality'] > 0 else 0,
    axis=1
)

print('\n--- Analysis of Normalized Degree Centrality Y ---')

# Top 10 of the new metric
top_10_normalized = temp_df.nlargest(10, 'normalized_degree_centrality_y')
print("Top 10 highest normalized_degree_centrality_y:")
display(top_10_normalized[['vertex', 'Label', 'normalized_degree_centrality_y']])

# Bottom 10 of the new metric
bottom_10_normalized = temp_df.nsmallest(10, 'normalized_degree_centrality_y')
print("Bottom 10 lowest normalized_degree_centrality_y:")
display(bottom_10_normalized[['vertex', 'Label', 'normalized_degree_centrality_y']])

# Filter temp_df for Layer 0 and Layer 1 nodes only
# 'layer0_1_nodes' is defined in a previous cell and contains the IDs of nodes in layers 0 and 1
temp_df_filtered = temp_df[temp_df['vertex'].isin(layer0_1_nodes)].copy()

# Calculate ranks for original and new metric on the filtered DataFrame
temp_df_filtered['rank_original'] = temp_df_filtered['degree_centrality_y'].rank(ascending=False)
temp_df_filtered['rank_normalized'] = temp_df_filtered['normalized_degree_centrality_y'].rank(ascending=False)

# Calculate rank change (original rank - new rank)
# A positive change means the rank improved (riser), negative means it dropped (faller)
temp_df_filtered['rank_change'] = temp_df_filtered['rank_original'] - temp_df_filtered['rank_normalized']

print('\n--- Top 3 Biggest Risers in Rank (Normalized Degree Centrality Y - Layers 0 & 1) ---')
# Risers have a more positive rank_change
risers = temp_df_filtered.nlargest(3, 'rank_change')
display(risers[['vertex', 'Label', 'degree_centrality_y', 'normalized_degree_centrality_y', 'rank_original', 'rank_normalized', 'rank_change']])

print('\n--- Top 3 Biggest Fallers in Rank (Normalized Degree Centrality Y - Layers 0 & 1) ---')
# Fallers have a more negative rank_change
fallers = temp_df_filtered.nsmallest(3, 'rank_change')
display(fallers[['vertex', 'Label', 'degree_centrality_y', 'normalized_degree_centrality_y', 'rank_original', 'rank_normalized', 'rank_change']])


--- Analysis of Normalized Degree Centrality Y ---
Top 10 highest normalized_degree_centrality_y:


,vertex,Label,normalized_degree_centrality_y
2595,2290076663,Vinit Ranjan,4.464418e+06
2592,41133423,K. Bose,2.917894e+06
2593,67311839,Jacob D. Moorman,2.793717e+06
2594,122992709,Yotam Yaniv,2.793717e+06
2591,145644823,Yansong Gao,1.410634e+06
2586,89804042,E. Demirkaya,1.376979e+06
2585,1557383324,Xianyang Zhang,1.140297e+06
2587,104093855,Xiaoyang Zhuo,1.033495e+06
2588,101465808,Antoine-Marie Bogso,1.033495e+06
2589,102499420,Suhang Dai,1.033495e+06


Bottom 10 lowest normalized_degree_centrality_y:


,vertex,Label,normalized_degree_centrality_y
2596,48876151,Mingyu Ding,0.0
2597,1581341133,S. Cohan,0.0
2598,2005667112,S. Chabra,0.0
2599,1712880,S. Tomasin,0.0
2600,2404961226,Ruixuan Li,0.0
2601,14256521,S. Chun,0.0
2602,144071961,S. Tiwari,0.0
2603,2349956736,Ryan Zhao,0.0
2604,153671941,S. Chander,0.0
2605,108554478,S. Chua,0.0



--- Top 3 Biggest Risers in Rank (Normalized Degree Centrality Y - Layers 0 & 1) ---


,vertex,Label,degree_centrality_y,normalized_degree_centrality_y,rank_original,rank_normalized,rank_change
1727,2067525096,Mathias Pohl,0.183489,2103.295196,562.0,37.0,525.0
2563,2381373459,Quentin Cormier,0.202814,216119.703462,522.0,4.0,518.0
1434,2258723043,Dylan Possamai,0.193906,1541.974105,554.0,43.0,511.0



--- Top 3 Biggest Fallers in Rank (Normalized Degree Centrality Y - Layers 0 & 1) ---


,vertex,Label,degree_centrality_y,normalized_degree_centrality_y,rank_original,rank_normalized,rank_change
0,47829900,Fan Yang,0.394815,1.031778,1.0,519.0,-518.0
1,38144094,Tiangang Zhang,0.353783,2.611040,2.0,518.0,-516.0
2,145784853,Xiao Han,0.348931,2.952747,3.0,516.0,-513.0


In [ ]:
# Initialize a list to store the reach for each Layer 0 professor
layer0_reach_data = []

# Get all Layer 0 node IDs
layer0_node_ids = nodes_df[nodes_df['Layer'] == 0]['Id'].tolist()

print("\n--- Calculating Reach for Layer 0 Professors ---")

for l0_node_id in layer0_node_ids:
    # Get the label (name) of the Layer 0 professor
    l0_professor_name = nodes_df[nodes_df['Id'] == l0_node_id]['Label'].iloc[0]

    # Step 1: Find Layer 1 neighbors of the current Layer 0 professor
    neighbors_of_l0 = list(G.neighbors(l0_node_id))
    l1_neighbors_of_l0 = [n_id for n_id in neighbors_of_l0 if node_to_layer.get(n_id) == 1]

    # Count edges from L0 to L1 (number of L1 neighbors)
    edges_l0_to_l1 = len(l1_neighbors_of_l0)

    # Step 2: For each Layer 1 neighbor, find its Layer 2 neighbors
    edges_l1_to_l2 = 0
    for l1_node_id in l1_neighbors_of_l0:
        neighbors_of_l1 = list(G.neighbors(l1_node_id))
        # Count edges from this L1 node to L2 nodes (number of L2 neighbors)
        edges_l1_to_l2 += len([n_id for n_id in neighbors_of_l1 if node_to_layer.get(n_id) == 2])

    # Calculate total reach for this Layer 0 professor
    total_reach = edges_l0_to_l1 + edges_l1_to_l2

    layer0_reach_data.append({
        'Layer 0 Professor ID': l0_node_id,
        'Professor Name': l0_professor_name,
        'Edges to L1 Professors': edges_l0_to_l1,
        'Edges from L1 to L2 Professors': edges_l1_to_l2,
        'Total Reach': total_reach
    })

# Convert the results to a DataFrame for better display
layer0_reach_df = pd.DataFrame(layer0_reach_data)

# Sort by total reach and display
layer0_reach_df_sorted = layer0_reach_df.sort_values(by='Total Reach', ascending=False)

print("\nReach of Layer 0 Professors (Sorted by Total Reach):")
display(layer0_reach_df_sorted)


--- Calculating Reach for Layer 0 Professors ---

Reach of Layer 0 Professors (Sorted by Total Reach):


,Layer 0 Professor ID,Professor Name,Edges to L1 Professors,Edges from L1 to L2 Professors,Total Reach
7,2266124380,Jianqing Fan,118,10990,11108
2,8386952,Bartolomeo Stellato,75,5404,5479
3,2258225151,Boris Hanin,66,4628,4694
15,1697413,Sanjeev R. Kulkarni,39,3077,3116
4,12890978,E. Rebrova,49,1468,1517
6,9051324,Jason M. Klusowski,37,1284,1321
8,2092062,John M. Mulvey,26,1249,1275
12,25681050,Matias D. Cattaneo,34,720,754
5,1825721854,H. Mete Soner,30,506,536
1,1792465,Amir Ali Ahmadi,10,523,533


In [ ]:
overall_average_degree_centrality = sum(nx.degree_centrality(G).values()) / G.number_of_nodes()
print(f"Average degree centrality for the entire graph: {overall_average_degree_centrality:.4f}")

Average degree centrality for the entire graph: 0.0001


In [ ]:
# Filter out cycles of length 3
cycles_len_3 = [c for c in cycles if len(c) == 3]

count_all_same_layer = 0
count_includes_layer2 = 0

# New counters for detailed same-layer breakdown
count_same_layer_0 = 0
count_same_layer_1 = 0
count_same_layer_2 = 0

for cycle in cycles_len_3:
    node1, node2, node3 = cycle

    layer1 = node_to_layer.get(node1)
    layer2 = node_to_layer.get(node2)
    layer3 = node_to_layer.get(node3)

    # Check if all nodes are from the same layer
    if layer1 == layer2 and layer2 == layer3:
        count_all_same_layer += 1
        if layer1 == 0:
            count_same_layer_0 += 1
        elif layer1 == 1:
            count_same_layer_1 += 1
        elif layer1 == 2:
            count_same_layer_2 += 1

    # Check if any node includes a node from Layer 2
    if 2 in [layer1, layer2, layer3]:
        count_includes_layer2 += 1

print(f"\n--- 3-Cycle Layer Analysis ---")
print(f"Total 3-cycles found: {len(cycles_len_3)}")
print(f"Number of 3-cycles with all nodes from the same layer: {count_all_same_layer}")
print(f"  - All nodes in Layer 0: {count_same_layer_0}")
print(f"  - All nodes in Layer 1: {count_same_layer_1}")
print(f"  - All nodes in Layer 2: {count_same_layer_2}")
print(f"Number of 3-cycles including at least one node from Layer 2: {count_includes_layer2}")


--- 3-Cycle Layer Analysis ---
Total 3-cycles found: 4263
Number of 3-cycles with all nodes from the same layer: 549
  - All nodes in Layer 0: 0
  - All nodes in Layer 1: 549
  - All nodes in Layer 2: 0
Number of 3-cycles including at least one node from Layer 2: 3342


In [ ]:
import networkx as nx

# Filter cycles by length 4 and 5
cycles_len_4 = [c for c in cycles if len(c) == 4]
cycles_len_5 = [c for c in cycles if len(c) == 5]

# Function to check for a 3-cycle within a larger cycle's subgraph
def contains_3_cycle(graph, cycle_nodes):
    if len(cycle_nodes) < 3:
        return False
    # Create a subgraph using only the nodes of the current cycle
    subgraph = graph.subgraph(cycle_nodes)
    # Find cycle basis for the subgraph and check for 3-cycles
    subgraph_cycles = nx.cycle_basis(subgraph)
    for sub_cycle in subgraph_cycles:
        if len(sub_cycle) == 3:
            return True
    return False

# Analyze 4-cycles
count_4_with_3_cycle = 0
for cycle_4 in cycles_len_4:
    if contains_3_cycle(G, cycle_4):
        count_4_with_3_cycle += 1

total_4_cycles = len(cycles_len_4)
percentage_4_with_3_cycle = (count_4_with_3_cycle / total_4_cycles * 100) if total_4_cycles > 0 else 0

print(f"\n--- 4-Cycle Analysis ---")
print(f"Total 4-cycles found: {total_4_cycles}")
print(f"Number of 4-cycles containing a 3-cycle: {count_4_with_3_cycle}")
print(f"Percentage of 4-cycles containing a 3-cycle: {percentage_4_with_3_cycle:.2f}%")

# Analyze 5-cycles
count_5_with_3_cycle = 0
for cycle_5 in cycles_len_5:
    if contains_3_cycle(G, cycle_5):
        count_5_with_3_cycle += 1

total_5_cycles = len(cycles_len_5)
percentage_5_with_3_cycle = (count_5_with_3_cycle / total_5_cycles * 100) if total_5_cycles > 0 else 0

print(f"\n--- 5-Cycle Analysis ---")
print(f"Total 5-cycles found: {total_5_cycles}")
print(f"Number of 5-cycles containing a 3-cycle: {count_5_with_3_cycle}")
print(f"Percentage of 5-cycles containing a 3-cycle: {percentage_5_with_3_cycle:.2f}%")


--- 4-Cycle Analysis ---
Total 4-cycles found: 315
Number of 4-cycles containing a 3-cycle: 0
Percentage of 4-cycles containing a 3-cycle: 0.00%

--- 5-Cycle Analysis ---
Total 5-cycles found: 183
Number of 5-cycles containing a 3-cycle: 0
Percentage of 5-cycles containing a 3-cycle: 0.00%


In [ ]:
# Reuse the contains_3_cycle function defined previously
# def contains_3_cycle(graph, cycle_nodes):
#     if len(cycle_nodes) < 3:
#         return False
#     subgraph = graph.subgraph(cycle_nodes)
#     subgraph_cycles = nx.cycle_basis(subgraph)
#     for sub_cycle in subgraph_cycles:
#         if len(sub_cycle) == 3:
#             return True
#     return False


print("\n--- Analysis of Cycles > 3 for Embedded 3-Cycles ---")

count_longer_cycles_with_3_cycle = 0
total_longer_cycles = 0

for cycle in cycles:
    if len(cycle) > 3:
        total_longer_cycles += 1
        if contains_3_cycle(G, cycle):
            count_longer_cycles_with_3_cycle += 1

percentage_longer_cycles_with_3_cycle = \
    (count_longer_cycles_with_3_cycle / total_longer_cycles * 100) if total_longer_cycles > 0 else 0

print(f"Total cycles with length greater than 3: {total_longer_cycles}")
print(f"Number of these cycles containing an embedded 3-cycle: {count_longer_cycles_with_3_cycle}")
print(f"Percentage of these cycles containing an embedded 3-cycle: {percentage_longer_cycles_with_3_cycle:.2f}%")


--- Analysis of Cycles > 3 for Embedded 3-Cycles ---
Total cycles with length greater than 3: 3358
Number of these cycles containing an embedded 3-cycle: 0
Percentage of these cycles containing an embedded 3-cycle: 0.00%
